# Comprobaciones de integridad del dataset jugador–partido

## Objetivo del notebook

Este notebook realiza las comprobaciones de calidad e integridad necesarias sobre el dataset jugador–partido obtenido tras la preparación y selección inicial de variables.

Se parte de `df_jugador_actual.csv` y se verifica, entre otros aspectos, la estructura recíproca de los encuentros, la coherencia entre jugador y rival, la consistencia de juegos y sets, la definición de `porcentaje_juegos_ganados`, la información temporal y la existencia de casos sin volumen competitivo observable.

Estas comprobaciones se realizan antes de construir las variables de carga competitiva acumulada, con el objetivo de garantizar que el dataset utilizado en las etapas posteriores mantiene una estructura consistente y temporalmente interpretable.

Tras las comprobaciones y la exclusión del único encuentro sin volumen competitivo observable, se genera `df_jugador_limpio.csv`, que constituye la entrada del siguiente bloque del pipeline dedicado a la construcción de las cargas acumuladas.

## 0. Librerías necesarias

En este bloque se importan las librerías que se van a utilizar durante la preparación de datos.

- `numpy`: operaciones numéricas y comprobaciones.
- `pandas`: carga, transformación y validación de tablas.
- `Path`: comprobación y gestión de rutas de los archivos generados.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
file_path = "df_jugador_actual.csv"

df_master = pd.read_csv(
    file_path,
    low_memory=False
)

df_master["fecha_torneo"] = pd.to_datetime(
    df_master["fecha_torneo"],
    errors="raise"
)

print("Dimensiones:", df_master.shape)

columnas_control_temporal = [
    "es_muestra_estudio",
    "es_buffer_carga",
    "es_desarrollo_modelo",
    "es_validacion_temporal"
]

assert set(columnas_control_temporal).issubset(df_master.columns)

display(
    df_master.groupby(
        [
            "match_year",
            "es_buffer_carga",
            "es_desarrollo_modelo",
            "es_validacion_temporal",
            "es_muestra_estudio"
        ]
    )
    .size()
    .rename("observaciones")
    .to_frame()
)


Dimensiones: (94032, 78)


,,,,,observaciones
match_year,es_buffer_carga,es_desarrollo_modelo,es_validacion_temporal,es_muestra_estudio,
2008,1,0,0,0,6204
2009,0,1,0,1,6138
2010,0,1,0,1,6026
2011,0,1,0,1,6000
2012,0,1,0,1,5980
2013,0,1,0,1,5846
2014,0,1,0,1,5748
2015,0,1,0,1,5866
2016,0,1,0,1,5840


## Comprobaciones varias

In [3]:
# Paso 3: comprobación general del DataFrame.

print("Número de filas:", len(df_master))
print("Número de columnas:", df_master.shape[1])
print("Número de jugadores:", df_master["jugador_id"].nunique())
print("Fecha mínima:", df_master["fecha_torneo"].min())
print("Fecha máxima:", df_master["fecha_torneo"].max())

print("\nValores nulos en campos esenciales:")
display(
    df_master[
        [
            "jugador_id",
            "rival_id",
            "fecha_torneo",
            "identificador_torneo",
            "numero_partido",
            "porcentaje_juegos_ganados"
            
        ]
    ].isna().sum()
)

Número de filas: 94032
Número de columnas: 78
Número de jugadores: 1816
Fecha mínima: 2007-12-31 00:00:00
Fecha máxima: 2024-12-18 00:00:00

Valores nulos en campos esenciales:


jugador_id                   0
rival_id                     0
fecha_torneo                 0
identificador_torneo         0
numero_partido               0
porcentaje_juegos_ganados    2
dtype: int64

In [4]:
# Comprobaciones estructurales.

assert len(df_master) > 0, (
    "El dataset está vacío."
)

assert df_master.columns.is_unique, (
    "Existen columnas duplicadas en el dataset."
)

columnas_minimas = [
    "identificador_torneo",
    "numero_partido",
    "jugador_id",
    "rival_id",
    "fecha_torneo",
    "match_year",
    "ronda",
    "ronda_encoded",
    "cabeza_serie",
    "cabeza_serie_rival",
    "total_juegos",
    "total_sets",
    "porcentaje_juegos_ganados",
    "es_muestra_estudio",
    "es_buffer_carga",
    "es_desarrollo_modelo",
    "es_validacion_temporal"
]

columnas_faltantes = [
    columna
    for columna in columnas_minimas
    if columna not in df_master.columns
]

assert not columnas_faltantes, (
    f"Faltan columnas mínimas necesarias: {columnas_faltantes}"
)

assert "BR" not in df_master["ronda"].dropna().unique()

assert df_master["ronda_encoded"].notna().all()

assert set(
    df_master["cabeza_serie"].dropna().unique()
).issubset({0, 1})

assert set(
    df_master["cabeza_serie_rival"].dropna().unique()
).issubset({0, 1})

assert (
    df_master["es_buffer_carga"]
    + df_master["es_desarrollo_modelo"]
    + df_master["es_validacion_temporal"]
).eq(1).all()

assert (
    df_master["es_desarrollo_modelo"]
    + df_master["es_validacion_temporal"]
).eq(df_master["es_muestra_estudio"]).all()

assert 2020 not in df_master["match_year"].unique()

assert df_master.loc[
    df_master["match_year"].isin([2008, 2021]),
    "es_buffer_carga"
].eq(1).all()

assert df_master.loc[
    df_master["match_year"].eq(2024),
    "es_validacion_temporal"
].eq(1).all()

objetivo = df_master["porcentaje_juegos_ganados"]

assert objetivo.dropna().between(0, 1).all()

print("Número de filas:", len(df_master))
print("Número de columnas:", df_master.shape[1])
print("Número de nulos en porcentaje_juegos_ganados:", objetivo.isna().sum())
print("Las comprobaciones iniciales se han superado.")

Número de filas: 94032
Número de columnas: 78
Número de nulos en porcentaje_juegos_ganados: 2
Las comprobaciones iniciales se han superado.


### 3.1. Construcción y validación del identificador de partido

 Se crea una copia de comprobación del dataset y se construye un identificador auxiliar de partido mediante la combinación de `identificador_torneo` y `numero_partido`.

El dataset está organizado a nivel jugador-partido, por lo que cada encuentro debe aparecer exactamente mediante dos observaciones. Disponer de un identificador estable es necesario para vincular ambas perspectivas, conservar su trazabilidad y mantenerlas en la misma partición temporal.

* Se evita mezclar observaciones pertenecientes a partidos diferentes, contar encuentros más de una vez o separar las dos perspectivas de un mismo partido durante la validación.

La columna auxiliar se crea únicamente en `df_check`. El DataFrame original `df_master` no se modifica.

In [5]:
# Crea una copia profunda destinada exclusivamente a las comprobaciones.
# df_master permanece como dataset original de esta fase.
df_check = df_master.copy(deep=True)


# Comprueba que las columnas necesarias para identificar los partidos existen.
columnas_identificacion = [
    "identificador_torneo",
    "numero_partido",
    "jugador_id",
    "rival_id"
]

columnas_faltantes = [
    columna
    for columna in columnas_identificacion
    if columna not in df_check.columns
]

assert not columnas_faltantes, (
    f"Faltan columnas necesarias para identificar los partidos: "
    f"{columnas_faltantes}"
)


# Comprueba que los campos que forman el identificador no contienen valores ausentes.
nulos_identificacion = (
    df_check[columnas_identificacion]
    .isna()
    .sum()
)

display(
    nulos_identificacion.to_frame(
        name="numero_nulos"
    )
)

assert nulos_identificacion.sum() == 0, (
    "Existen valores ausentes en las columnas utilizadas "
    "para identificar los partidos."
)


# Normaliza numero_partido como número entero.
# errors='raise' detiene la ejecución si existe algún valor no interpretable.
numero_partido_normalizado = (
    pd.to_numeric(
        df_check["numero_partido"],
        errors="raise"
    )
    .astype("Int64")
    .astype("string")
)


# Construye un identificador auxiliar único para cada encuentro.
df_check["_id_partido"] = (
    df_check["identificador_torneo"]
    .astype("string")
    .str.strip()
    + "__"
    + numero_partido_normalizado
)


# Comprueba que la nueva columna no se ha añadido accidentalmente a df_master.
assert "_id_partido" not in df_master.columns, (
    "df_master ha sido modificado accidentalmente."
)


# Comprueba que el identificador construido no contiene valores ausentes.
assert df_check["_id_partido"].notna().all(), (
    "Se han generado identificadores de partido ausentes."
)


# Calcula el número de registros, partidos y jugadores.
resumen_identificacion = pd.Series(
    {
        "numero_observaciones": len(df_check),
        "numero_partidos_identificados": df_check["_id_partido"].nunique(),
        "numero_jugadores": df_check["jugador_id"].nunique(),
        "duplicados_jugador_partido": df_check.duplicated(
            subset=["_id_partido", "jugador_id"]
        ).sum()
    },
    name="valor"
)

display(resumen_identificacion.to_frame())


# Comprueba que un mismo jugador no aparece repetido
# dentro del mismo partido.
assert resumen_identificacion["duplicados_jugador_partido"] == 0, (
    "Se han encontrado combinaciones jugador-partido duplicadas."
)


assert resumen_identificacion["numero_partidos_identificados"] * 2 == len(df_check)


print(
    "Identificador de partido construido correctamente. "
    "No existen duplicados jugador-partido."
)

,numero_nulos
identificador_torneo,0
numero_partido,0
jugador_id,0
rival_id,0


,valor
numero_observaciones,94032
numero_partidos_identificados,47016
numero_jugadores,1816
duplicados_jugador_partido,0


Identificador de partido construido correctamente. No existen duplicados jugador-partido.


### 3.2. Validación de las dos perspectivas de cada partido

**Qué hago.** Compruebo que cada identificador de partido contiene exactamente dos observaciones, correspondientes a dos jugadores distintos. También verifico que la relación entre jugador y rival es recíproca: si en una fila aparece el jugador A frente al jugador B, en la segunda fila del mismo partido debe aparecer el jugador B frente al jugador A.

**Por qué lo hago.** Necesito garantizar que la transformación al formato jugador–partido conserva correctamente las dos perspectivas de cada encuentro. Esta estructura será fundamental para incorporar posteriormente la carga competitiva del rival y para mantener ambas observaciones dentro de la misma partición temporal.

**Riesgo metodológico controlado.** Evito trabajar con partidos incompletos, registros duplicados, autoenfrentamientos o asociaciones incorrectas entre jugadores y rivales.

In [6]:
# Compruebo que el identificador creado en el paso anterior está disponible.
assert "_id_partido" in df_check.columns, (
    "No encuentro la columna _id_partido. "
    "Debo ejecutar primero el apartado 3.1."
)


# Resumo la estructura de cada partido.
resumen_estructura_partidos = (
    df_check
    .groupby("_id_partido", sort=False)
    .agg(
        numero_registros=("jugador_id", "size"),
        numero_jugadores=("jugador_id", "nunique"),
        numero_rivales=("rival_id", "nunique")
    )
)


# Compruebo si alguna fila presenta al mismo jugador como su propio rival.
autoenfrentamientos = df_check[
    df_check["jugador_id"].eq(df_check["rival_id"])
].copy()


# Obtengo el conjunto de jugadores y el conjunto de rivales de cada partido.
jugadores_por_partido = (
    df_check
    .groupby("_id_partido", sort=False)["jugador_id"]
    .agg(frozenset)
)

rivales_por_partido = (
    df_check
    .groupby("_id_partido", sort=False)["rival_id"]
    .agg(frozenset)
)


# Compruebo la reciprocidad.
# En cada partido, el conjunto de jugadores debe coincidir
# exactamente con el conjunto de rivales.
reciprocidad_correcta = jugadores_por_partido.eq(
    rivales_por_partido
)


# Resumo todas las posibles anomalías.
resumen_validacion_perspectivas = pd.Series(
    {
        "partidos_con_numero_registros_distinto_de_2": (
            resumen_estructura_partidos["numero_registros"].ne(2).sum()
        ),
        "partidos_con_numero_jugadores_distinto_de_2": (
            resumen_estructura_partidos["numero_jugadores"].ne(2).sum()
        ),
        "partidos_con_numero_rivales_distinto_de_2": (
            resumen_estructura_partidos["numero_rivales"].ne(2).sum()
        ),
        "filas_con_jugador_igual_a_rival": len(autoenfrentamientos),
        "partidos_sin_reciprocidad": (
            (~reciprocidad_correcta).sum()
        )
    },
    name="numero_anomalias"
)

display(resumen_validacion_perspectivas.to_frame())


# Muestro ejemplos únicamente si encuentro alguna anomalía.
ids_problematicos = set(
    resumen_estructura_partidos.index[
        resumen_estructura_partidos["numero_registros"].ne(2)
        | resumen_estructura_partidos["numero_jugadores"].ne(2)
        | resumen_estructura_partidos["numero_rivales"].ne(2)
    ]
)

ids_problematicos.update(
    reciprocidad_correcta.index[
        ~reciprocidad_correcta
    ].tolist()
)

ids_problematicos.update(
    autoenfrentamientos["_id_partido"].tolist()
)


if ids_problematicos:
    print("He encontrado partidos problemáticos. Muestro algunos ejemplos:")

    display(
        df_check.loc[
            df_check["_id_partido"].isin(ids_problematicos),
            [
                "_id_partido",
                "jugador_id",
                "jugador_nombre",
                "rival_id",
                "rival_nombre"
            ]
        ]
        .sort_values(["_id_partido", "jugador_id"])
        .head(20)
    )


# Detengo la ejecución si alguna condición estructural no se cumple.
assert resumen_estructura_partidos["numero_registros"].eq(2).all(), (
    "He encontrado partidos que no contienen exactamente dos registros."
)

assert resumen_estructura_partidos["numero_jugadores"].eq(2).all(), (
    "He encontrado partidos que no contienen exactamente dos jugadores distintos."
)

assert resumen_estructura_partidos["numero_rivales"].eq(2).all(), (
    "He encontrado partidos que no contienen exactamente dos rivales distintos."
)

assert autoenfrentamientos.empty, (
    "He encontrado filas en las que el jugador coincide con su rival."
)

assert reciprocidad_correcta.all(), (
    "He encontrado partidos en los que las perspectivas "
    "de jugador y rival no son recíprocas."
)


print(
    "He validado correctamente las dos perspectivas de cada partido. "
    "Todos los encuentros contienen dos jugadores distintos "
    "y la relación jugador-rival es recíproca."
)

,numero_anomalias
partidos_con_numero_registros_distinto_de_2,0
partidos_con_numero_jugadores_distinto_de_2,0
partidos_con_numero_rivales_distinto_de_2,0
filas_con_jugador_igual_a_rival,0
partidos_sin_reciprocidad,0


He validado correctamente las dos perspectivas de cada partido. Todos los encuentros contienen dos jugadores distintos y la relación jugador-rival es recíproca.


### 3.3. Coherencia interna de las dos perspectivas del partido

**Qué hago.** Compruebo que las dos observaciones de cada partido comparten el mismo contexto competitivo y las mismas estadísticas globales del encuentro. También verifico que las variables de rendimiento se intercambian correctamente entre ambas perspectivas.

**Por qué lo hago.** Necesito garantizar que las dos filas de un partido representan el mismo encuentro y que únicamente cambia la perspectiva desde la que se observa. Esta coherencia será necesaria para construir correctamente la carga histórica del jugador y del rival.

**Riesgo metodológico controlado.** Evito contabilizar un mismo partido con valores diferentes de juegos, sets o duración y detecto posibles errores en la transformación de ganador y perdedor al formato jugador–partido.

In [7]:
# Compruebo que he ejecutado previamente la construcción
# y validación del identificador de partido.
assert "_id_partido" in df_check.columns, (
    "No encuentro la columna _id_partido. "
    "Debo ejecutar primero los apartados anteriores."
)


# Defino las variables que deben presentar exactamente
# el mismo valor en las dos perspectivas del encuentro.
columnas_compartidas = [
    "fecha_torneo",
    "nombre_torneo",
    "superficie",
    "nivel_torneo",
    "ronda",
    "ronda_encoded",
    "es_round_robin",
    "tamano_cuadro",
    "numero_maximo_sets",
    "match_year",
    "es_muestra_estudio",
    "es_buffer_carga",
    "es_desarrollo_modelo",
    "es_validacion_temporal",
    "marcador_partido",
    "duracion_partido_minutos",
    "total_juegos",
    "total_sets"
]


# Compruebo que todas las columnas necesarias están disponibles.
columnas_faltantes = [
    columna
    for columna in columnas_compartidas
    if columna not in df_check.columns
]

assert not columnas_faltantes, (
    "No encuentro algunas columnas compartidas: "
    f"{columnas_faltantes}"
)


# Calculo cuántos valores diferentes aparecen dentro de cada partido.
# dropna=False permite tratar un valor ausente como una categoría,
# de modo que también detecto casos con un valor presente y otro ausente.
n_unicos_por_partido = (
    df_check
    .groupby("_id_partido", sort=False)[columnas_compartidas]
    .nunique(dropna=False)
)


# Para cada variable, cuento en cuántos partidos aparecen
# valores diferentes entre sus dos perspectivas.
anomalias_compartidas = (
    n_unicos_por_partido
    .gt(1)
    .sum()
    .rename("numero_partidos_incoherentes")
    .to_frame()
)

display(anomalias_compartidas)


# Identifico los partidos con alguna incoherencia.
mascara_partidos_incoherentes = n_unicos_por_partido.gt(1).any(axis=1)

ids_partidos_incoherentes = (
    n_unicos_por_partido
    .index[mascara_partidos_incoherentes]
    .tolist()
)


# Muestro ejemplos únicamente si encuentro anomalías.
if ids_partidos_incoherentes:
    print(
        "He encontrado partidos con diferencias "
        "en variables que deberían ser compartidas."
    )

    display(
        df_check.loc[
            df_check["_id_partido"].isin(
                ids_partidos_incoherentes[:10]
            ),
            ["_id_partido", "jugador_nombre", "rival_nombre"]
            + columnas_compartidas
        ]
        .sort_values(["_id_partido", "jugador_id"])
    )


# Detengo la ejecución si alguna variable compartida
# presenta valores diferentes dentro de un mismo partido.
assert not ids_partidos_incoherentes, (
    "He encontrado partidos con valores diferentes "
    "en variables compartidas."
)


print(
    "He comprobado que las variables contextuales y globales "
    "coinciden en las dos perspectivas de todos los partidos."
)

,numero_partidos_incoherentes
fecha_torneo,0
nombre_torneo,0
superficie,0
nivel_torneo,0
ronda,0
ronda_encoded,0
es_round_robin,0
tamano_cuadro,0
numero_maximo_sets,0
match_year,0


He comprobado que las variables contextuales y globales coinciden en las dos perspectivas de todos los partidos.


In [8]:
import numpy as np


# Defino las variables necesarias para comprobar
# la complementariedad entre las dos perspectivas.
columnas_complementarias = [
    "juegos_ganados",
    "juegos_perdidos",
    "sets_ganados",
    "sets_perdidos",
    "total_juegos",
    "total_sets",
    "porcentaje_juegos_ganados",
]


# Compruebo que todas las columnas están disponibles.
columnas_faltantes = [
    columna
    for columna in columnas_complementarias
    if columna not in df_check.columns
]

assert not columnas_faltantes, (
    "No encuentro algunas columnas necesarias: "
    f"{columnas_faltantes}"
)


# Ordeno las dos perspectivas de cada partido de forma estable.
# El orden elegido no tiene interpretación deportiva;
# únicamente permite colocar las dos filas en columnas diferentes.
df_pares = (
    df_check[
        ["_id_partido", "jugador_id", "jugador_nombre"]
        + columnas_complementarias
    ]
    .sort_values(["_id_partido", "jugador_id"])
    .copy()
)


# Numero las dos perspectivas como 0 y 1.
df_pares["_perspectiva"] = (
    df_pares
    .groupby("_id_partido", sort=False)
    .cumcount()
)


# Transformo las dos filas de cada partido en una sola fila,
# manteniendo las dos perspectivas en columnas separadas.
df_partidos_pareados = df_pares.pivot(
    index="_id_partido",
    columns="_perspectiva",
    values=columnas_complementarias
)


# Creo una función auxiliar para extraer variables
# y convertirlas de forma segura a valores numéricos.
def extraer_numerica(nombre_variable, perspectiva):
    return (
        pd.to_numeric(
            df_partidos_pareados[
                (nombre_variable, perspectiva)
            ],
            errors="raise"
        )
        .astype("float64")
    )


# Creo una función para comparar valores numéricos,
# admitiendo pequeñas diferencias de redondeo
# y valores ausentes simétricos.
def son_iguales(a, b, tolerancia=1e-10):
    return pd.Series(
        np.isclose(
            a,
            b,
            atol=tolerancia,
            rtol=0,
            equal_nan=True
        ),
        index=a.index
    )


# =========================================================
# Extraigo las dos perspectivas de cada variable.
# =========================================================

juegos_ganados_0 = extraer_numerica("juegos_ganados", 0)
juegos_ganados_1 = extraer_numerica("juegos_ganados", 1)

juegos_perdidos_0 = extraer_numerica("juegos_perdidos", 0)
juegos_perdidos_1 = extraer_numerica("juegos_perdidos", 1)

sets_ganados_0 = extraer_numerica("sets_ganados", 0)
sets_ganados_1 = extraer_numerica("sets_ganados", 1)

sets_perdidos_0 = extraer_numerica("sets_perdidos", 0)
sets_perdidos_1 = extraer_numerica("sets_perdidos", 1)

porcentaje_ganados_0 = extraer_numerica(
    "porcentaje_juegos_ganados", 0
)

porcentaje_ganados_1 = extraer_numerica(
    "porcentaje_juegos_ganados", 1
)


# =========================================================
# Reciprocidad de juegos y sets.
# =========================================================

# Los juegos ganados por cada jugador deben coincidir
# con los juegos perdidos por su rival.
juegos_intercambiados = (
    son_iguales(juegos_ganados_0, juegos_perdidos_1)
    & son_iguales(juegos_ganados_1, juegos_perdidos_0)
)


# Compruebo la misma relación para los sets.
sets_intercambiados = (
    son_iguales(sets_ganados_0, sets_perdidos_1)
    & son_iguales(sets_ganados_1, sets_perdidos_0)
)


# =========================================================
# Coherencia de porcentaje_juegos_ganados.
# =========================================================

# Si el objetivo falta para una perspectiva,
# también debe faltar para la otra.
nulos_objetivo_simetricos = (
    porcentaje_ganados_0.isna()
    .eq(porcentaje_ganados_1.isna())
)


# Identifico los partidos en los que el objetivo
# está disponible en ambas perspectivas.
objetivo_completo = (
    porcentaje_ganados_0.notna()
    & porcentaje_ganados_1.notna()
)


# En los partidos completos, los porcentajes de juegos ganados
# de ambas perspectivas deben sumar exactamente uno,
# salvo diferencias numéricas despreciables.
porcentajes_entre_perspectivas = pd.Series(
    True,
    index=df_partidos_pareados.index
)

porcentajes_entre_perspectivas.loc[objetivo_completo] = (
    son_iguales(
        porcentaje_ganados_0.loc[objetivo_completo]
        + porcentaje_ganados_1.loc[objetivo_completo],
        pd.Series(
            1.0,
            index=porcentaje_ganados_0.loc[
                objetivo_completo
            ].index
        )
    )
)


# =========================================================
# Coherencia interna de juegos, sets y objetivo.
# =========================================================

# Compruebo directamente en cada observación que los totales
# coinciden con la suma de sus componentes.
total_juegos_coherente = son_iguales(
    pd.to_numeric(
        df_check["total_juegos"],
        errors="raise"
    ).astype("float64"),
    (
        pd.to_numeric(
            df_check["juegos_ganados"],
            errors="raise"
        ).astype("float64")
        +
        pd.to_numeric(
            df_check["juegos_perdidos"],
            errors="raise"
        ).astype("float64")
    )
)


total_sets_coherente = son_iguales(
    pd.to_numeric(
        df_check["total_sets"],
        errors="raise"
    ).astype("float64"),
    (
        pd.to_numeric(
            df_check["sets_ganados"],
            errors="raise"
        ).astype("float64")
        +
        pd.to_numeric(
            df_check["sets_perdidos"],
            errors="raise"
        ).astype("float64")
    )
)


# Compruebo que porcentaje_juegos_ganados coincide
# con juegos_ganados / total_juegos cuando puede calcularse.
total_juegos = pd.to_numeric(
    df_check["total_juegos"],
    errors="raise"
).astype("float64")

juegos_ganados = pd.to_numeric(
    df_check["juegos_ganados"],
    errors="raise"
).astype("float64")

porcentaje_guardado = pd.to_numeric(
    df_check["porcentaje_juegos_ganados"],
    errors="raise"
).astype("float64")


mascara_porcentaje_calculable = (
    total_juegos.notna()
    & total_juegos.gt(0)
    & juegos_ganados.notna()
    & porcentaje_guardado.notna()
)


porcentaje_recalculado = (
    juegos_ganados.loc[mascara_porcentaje_calculable]
    /
    total_juegos.loc[mascara_porcentaje_calculable]
)


porcentaje_correctamente_calculado = son_iguales(
    porcentaje_guardado.loc[mascara_porcentaje_calculable],
    porcentaje_recalculado
)


# =========================================================
# Resumen de anomalías.
# =========================================================

resumen_coherencia_partido = pd.Series(
    {
        "partidos_con_juegos_no_intercambiados": (
            (~juegos_intercambiados).sum()
        ),
        "partidos_con_sets_no_intercambiados": (
            (~sets_intercambiados).sum()
        ),
        "partidos_con_nulos_objetivo_asimetricos": (
            (~nulos_objetivo_simetricos).sum()
        ),
        "partidos_completos_con_porcentajes_no_complementarios": (
            (~porcentajes_entre_perspectivas).sum()
        ),
        "filas_con_total_juegos_incoherente": (
            (~total_juegos_coherente).sum()
        ),
        "filas_con_total_sets_incoherente": (
            (~total_sets_coherente).sum()
        ),
        "filas_con_porcentaje_mal_calculado": (
            (~porcentaje_correctamente_calculado).sum()
        ),
        "partidos_con_objetivo_ausente": (
            (~objetivo_completo).sum()
        )
    },
    name="numero_anomalias"
)

display(resumen_coherencia_partido.to_frame())


# =========================================================
# Identificación del objetivo ausente.
# =========================================================

filas_objetivo_ausente = df_check.loc[
    df_check["porcentaje_juegos_ganados"].isna(),
    [
        "_id_partido",
        "fecha_torneo",
        "nombre_torneo",
        "jugador_nombre",
        "rival_nombre",
        "marcador_partido",
        "juegos_ganados",
        "juegos_perdidos",
        "total_juegos"
    ]
]


if not filas_objetivo_ausente.empty:
    print(
        "He identificado las siguientes observaciones "
        "con porcentaje de juegos ganados ausente:"
    )

    display(filas_objetivo_ausente)


# =========================================================
# Validaciones.
# =========================================================

assert juegos_intercambiados.all(), (
    "He encontrado partidos en los que los juegos ganados "
    "y perdidos no se intercambian correctamente."
)

assert sets_intercambiados.all(), (
    "He encontrado partidos en los que los sets ganados "
    "y perdidos no se intercambian correctamente."
)

assert nulos_objetivo_simetricos.all(), (
    "He encontrado partidos en los que el objetivo falta "
    "solo en una de las dos perspectivas."
)

assert porcentajes_entre_perspectivas.all(), (
    "He encontrado partidos completos cuyos porcentajes "
    "de juegos ganados no suman uno."
)

assert total_juegos_coherente.all(), (
    "He encontrado observaciones con un total "
    "de juegos incoherente."
)

assert total_sets_coherente.all(), (
    "He encontrado observaciones con un total "
    "de sets incoherente."
)

assert porcentaje_correctamente_calculado.all(), (
    "He encontrado porcentajes de juegos ganados "
    "que no coinciden con su cálculo."
)


print(
    "He validado correctamente la coherencia interna "
    "y la complementariedad de las dos perspectivas."
)

,numero_anomalias
partidos_con_juegos_no_intercambiados,0
partidos_con_sets_no_intercambiados,0
partidos_con_nulos_objetivo_asimetricos,0
partidos_completos_con_porcentajes_no_complementarios,0
filas_con_total_juegos_incoherente,0
filas_con_total_sets_incoherente,0
filas_con_porcentaje_mal_calculado,0
partidos_con_objetivo_ausente,1


He identificado las siguientes observaciones con porcentaje de juegos ganados ausente:


,_id_partido,fecha_torneo,nombre_torneo,jugador_nombre,rival_nombre,marcador_partido,juegos_ganados,juegos_perdidos,total_juegos
30598,2013-338__14,2013-01-07,Sydney,Julien Benneteau,Radek Stepanek,RET,0,0,0
30599,2013-338__14,2013-01-07,Sydney,Radek Stepanek,Julien Benneteau,RET,0,0,0


He validado correctamente la coherencia interna y la complementariedad de las dos perspectivas.


### 3.4. Validación de la coherencia temporal

**Qué hago.** Compruebo que cada identificador de torneo está asociado a una única fecha de referencia, que `fecha_torneo` puede interpretarse correctamente como una fecha cronológica y que `match_year` identifica de forma coherente la temporada competitiva asignada a cada encuentro.

**Por qué lo hago.** La construcción posterior de variables de carga acumulada dependerá del orden cronológico de los torneos y, en el caso de las acumulaciones por temporada, de la asignación mediante `match_year`. Por ello, es necesario distinguir entre el año natural de `fecha_torneo` y la temporada competitiva a la que pertenece el torneo.

**Riesgo metodológico controlado.** Evito ordenar incorrectamente los torneos, mezclar temporadas o construir acumulaciones utilizando referencias temporales inconsistentes. En esta investigación, `fecha_torneo` se utiliza como fecha de referencia cronológica del torneo, mientras que `match_year` determina la temporada competitiva.

In [9]:
# Compruebo que las columnas temporales necesarias están disponibles.
columnas_temporales = [
    "identificador_torneo",
    "fecha_torneo",
    "match_year"
]

columnas_faltantes = [
    columna
    for columna in columnas_temporales
    if columna not in df_check.columns
]

assert not columnas_faltantes, (
    "No encuentro algunas columnas temporales necesarias: "
    f"{columnas_faltantes}"
)


# Convierto fecha_torneo a formato datetime de manera controlada.
# Creo primero una serie auxiliar para detectar posibles errores
# antes de sustituir la columna existente en df_check.
fecha_torneo_convertida = pd.to_datetime(
    df_check["fecha_torneo"],
    errors="coerce"
)


# Identifico las fechas que no han podido convertirse.
fechas_originales_no_nulas = df_check["fecha_torneo"].notna()

fechas_no_convertibles = (
    fechas_originales_no_nulas
    & fecha_torneo_convertida.isna()
)

numero_fechas_no_convertibles = int(
    fechas_no_convertibles.sum()
)


if numero_fechas_no_convertibles > 0:
    print(
        "He encontrado valores de fecha que no pueden "
        "convertirse correctamente:"
    )

    display(
        df_check.loc[
            fechas_no_convertibles,
            [
                "identificador_torneo",
                "fecha_torneo",
                "match_year"
            ]
        ].head(20)
    )


assert numero_fechas_no_convertibles == 0, (
    "Existen valores de fecha_torneo que no pueden "
    "interpretarse como fechas."
)


# Sustituyo la columna únicamente después de validar la conversión.
df_check["fecha_torneo"] = fecha_torneo_convertida


# Compruebo que no existen fechas ausentes.
numero_fechas_ausentes = int(
    df_check["fecha_torneo"].isna().sum()
)

assert numero_fechas_ausentes == 0, (
    "Existen valores ausentes en fecha_torneo."
)


# Compruebo cuántas fechas distintas tiene cada torneo.
fechas_distintas_por_torneo = (
    df_check
    .groupby("identificador_torneo", sort=False)["fecha_torneo"]
    .nunique(dropna=False)
)


# Identifico los torneos asociados a más de una fecha.
torneos_con_varias_fechas = (
    fechas_distintas_por_torneo[
        fechas_distintas_por_torneo.ne(1)
    ]
)


if not torneos_con_varias_fechas.empty:
    print(
        "He encontrado torneos asociados a más de una fecha:"
    )

    display(
        df_check.loc[
            df_check["identificador_torneo"].isin(
                torneos_con_varias_fechas.index
            ),
            [
                "identificador_torneo",
                "nombre_torneo",
                "fecha_torneo",
                "match_year"
            ]
        ]
        .drop_duplicates()
        .sort_values(
            ["identificador_torneo", "fecha_torneo"]
        )
        .head(30)
    )


# Convierto match_year a formato numérico entero.
match_year_numerico = pd.to_numeric(
    df_check["match_year"],
    errors="coerce"
)


# Identifico valores de match_year no convertibles.
match_year_no_convertible = (
    df_check["match_year"].notna()
    & match_year_numerico.isna()
)

assert not match_year_no_convertible.any(), (
    "Existen valores de match_year que no pueden "
    "interpretarse como años."
)


# Compruebo que match_year no contiene valores ausentes.
assert match_year_numerico.notna().all(), (
    "Existen valores ausentes en match_year."
)


# Extraigo el año de la fecha y lo comparo con match_year.
anio_fecha_torneo = df_check["fecha_torneo"].dt.year

coincidencia_anio = (
    match_year_numerico.astype("int64")
    .eq(anio_fecha_torneo)
)


# Muestro ejemplos únicamente si encuentro diferencias.
if not coincidencia_anio.all():
    print(
        "He encontrado observaciones en las que match_year "
        "no coincide con el año de fecha_torneo:"
    )

    display(
        df_check.loc[
            ~coincidencia_anio,
            [
                "identificador_torneo",
                "fecha_torneo",
                "match_year"
            ]
        ]
        .drop_duplicates()
        .head(20)
    )


# Obtengo la lista ordenada de temporadas disponibles.
temporadas_observadas = sorted(
    anio_fecha_torneo.unique().tolist()
)

temporadas_esperadas = (
    list(range(2009, 2020))
    + list(range(2022, 2025))
)


# Calculo un resumen temporal general.
resumen_temporal = pd.Series(
    {
        "fecha_minima": df_check["fecha_torneo"].min(),
        "fecha_maxima": df_check["fecha_torneo"].max(),
        "numero_temporadas": len(temporadas_observadas),
        "numero_torneos": (
            df_check["identificador_torneo"].nunique()
        ),
        "torneos_con_fecha_no_unica": (
            len(torneos_con_varias_fechas)
        ),
        "filas_con_match_year_incoherente": (
            (~coincidencia_anio).sum()
        ),
        "observaciones_2020": (
            anio_fecha_torneo.eq(2020).sum()
        ),
        "observaciones_2021": (
            anio_fecha_torneo.eq(2021).sum()
        )
    },
    name="valor"
)

display(resumen_temporal.to_frame())


print("Años naturales observados en fecha_torneo:")
print(temporadas_observadas)


# Muestro también el número de observaciones y torneos por temporada.
resumen_por_temporada = (
    df_check
    .assign(
        temporada_fecha=anio_fecha_torneo
    )
    .groupby("temporada_fecha")
    .agg(
        numero_observaciones=(
            "_id_partido",
            "size"
        ),
        numero_partidos=(
            "_id_partido",
            "nunique"
        ),
        numero_torneos=(
            "identificador_torneo",
            "nunique"
        ),
        numero_jugadores=(
            "jugador_id",
            "nunique"
        )
    )
)

display(resumen_por_temporada)


# Assertions finales de coherencia temporal.
assert torneos_con_varias_fechas.empty, (
    "He encontrado torneos asociados a más de una fecha."
)



temporadas_esperadas_match_year = (
    [2008]
    + list(range(2009, 2020))
    + [2021, 2022, 2023, 2024]
)

temporadas_observadas_match_year = sorted(
    df_check["match_year"].unique().tolist()
)

assert temporadas_observadas_match_year == temporadas_esperadas_match_year, (
    "Las temporadas observadas no coinciden con el dataset operativo esperado."
)

assert 2020 not in temporadas_observadas_match_year

assert df_check.loc[
    df_check["match_year"].isin([2008, 2021]),
    "es_buffer_carga"
].eq(1).all()

assert df_check.loc[
    df_check["match_year"].eq(2024),
    "es_validacion_temporal"
].eq(1).all()


print(
    "He validado correctamente la coherencia temporal. "
    "Cada torneo tiene una única fecha de referencia, "
    "los desfases entre fecha_torneo y match_year han sido auditados, "
    "y las temporadas operativas observadas son las esperadas."
)

He encontrado observaciones en las que match_year no coincide con el año de fecha_torneo:


,identificador_torneo,fecha_torneo,match_year
0,2008-339,2007-12-31,2008
62,2008-451,2007-12-31,2008
120,2008-891,2007-12-31,2008
30348,2013-339,2012-12-30,2013
30402,2013-451,2012-12-31,2013
30464,2013-891,2012-12-31,2013
36194,2014-339,2013-12-29,2014
36248,2014-451,2013-12-30,2014
36310,2014-891,2013-12-30,2014
65196,2019-0451,2018-12-31,2019


,valor
fecha_minima,2007-12-31 00:00:00
fecha_maxima,2024-12-18 00:00:00
numero_temporadas,17
numero_torneos,2272
torneos_con_fecha_no_unica,0
filas_con_match_year_incoherente,688
observaciones_2020,0
observaciones_2021,5424


Años naturales observados en fecha_torneo:
[2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2021, 2022, 2023, 2024]


,numero_observaciones,numero_partidos,numero_torneos,numero_jugadores
temporada_fecha,,,,
2007,182,91,3,96
2008,6022,3011,146,489
2009,6138,3069,148,463
2010,6026,3013,148,472
2011,6000,3000,147,459
2012,6150,3075,150,459
2013,5844,2922,147,456
2014,5580,2790,140,425
2015,5866,2933,147,429


He validado correctamente la coherencia temporal. Cada torneo tiene una única fecha de referencia, los desfases entre fecha_torneo y match_year han sido auditados, y las temporadas operativas observadas son las esperadas.


### 3.4.1. Diferencia entre el año natural y la temporada competitiva

He encontrado algunas observaciones en las que `match_year` no coincide con el año natural de `fecha_torneo`. Antes de considerar estas diferencias como errores, analizo su patrón.

`fecha_torneo` representa la fecha de referencia cronológica del torneo, mientras que `match_year` identifica la temporada competitiva a la que fue asignado. Algunos torneos pertenecientes a una temporada pueden comenzar durante los últimos días de diciembre del año natural anterior.

Compruebo que las diferencias se limitan a esta situación de cambio de año y que no existen desfases temporales de otra naturaleza.

In [10]:
# Extraigo el año natural, el mes y el día de la fecha del torneo.
df_check["anio_fecha_torneo"] = df_check["fecha_torneo"].dt.year
df_check["mes_fecha_torneo"] = df_check["fecha_torneo"].dt.month
df_check["dia_fecha_torneo"] = df_check["fecha_torneo"].dt.day


# Convierto match_year a entero para trabajar de forma consistente.
df_check["match_year"] = pd.to_numeric(
    df_check["match_year"],
    errors="raise"
).astype("int64")


# Calculo el desfase entre la temporada asignada
# y el año natural de la fecha.
df_check["desfase_anio_temporada"] = (
    df_check["match_year"]
    - df_check["anio_fecha_torneo"]
)


# Selecciono únicamente las observaciones con diferencia.
df_desfase_temporal = df_check.loc[
    df_check["desfase_anio_temporada"].ne(0)
].copy()


# Resumo el número de observaciones, partidos y torneos afectados.
resumen_desfase_temporal = pd.Series(
    {
        "observaciones_con_desfase": len(df_desfase_temporal),
        "partidos_con_desfase": (
            df_desfase_temporal["_id_partido"].nunique()
        ),
        "torneos_con_desfase": (
            df_desfase_temporal[
                "identificador_torneo"
            ].nunique()
        ),
        "desfase_minimo": (
            df_desfase_temporal[
                "desfase_anio_temporada"
            ].min()
        ),
        "desfase_maximo": (
            df_desfase_temporal[
                "desfase_anio_temporada"
            ].max()
        ),
        "mes_minimo": (
            df_desfase_temporal[
                "mes_fecha_torneo"
            ].min()
        ),
        "mes_maximo": (
            df_desfase_temporal[
                "mes_fecha_torneo"
            ].max()
        ),
        "dia_minimo": (
            df_desfase_temporal[
                "dia_fecha_torneo"
            ].min()
        ),
        "dia_maximo": (
            df_desfase_temporal[
                "dia_fecha_torneo"
            ].max()
        )
    },
    name="valor"
)

display(resumen_desfase_temporal.to_frame())

# =========================================================
# Validación del patrón de desfase temporal
# =========================================================

# Las diferencias entre match_year y el año natural de
# fecha_torneo solo son admisibles cuando el torneo pertenece
# a la temporada siguiente y comienza en diciembre.

assert (
    df_desfase_temporal["desfase_anio_temporada"]
    .eq(1)
    .all()
), (
    "He encontrado desfases entre match_year y fecha_torneo "
    "distintos de +1 año."
)

assert (
    df_desfase_temporal["mes_fecha_torneo"]
    .eq(12)
    .all()
), (
    "He encontrado desfases temporales fuera del mes de diciembre."
)

# Compruebo también que todas las observaciones del dataset
# presentan únicamente uno de los dos patrones válidos:
# año natural coincidente o temporada siguiente.
patron_temporal_valido = (
    df_check["desfase_anio_temporada"].eq(0)
    |
    (
        df_check["desfase_anio_temporada"].eq(1)
        & df_check["mes_fecha_torneo"].eq(12)
    )
)

assert patron_temporal_valido.all(), (
    "Existen observaciones cuyo patrón temporal no corresponde "
    "ni al año natural ni al cambio de temporada en diciembre."
)

print(
    "Los desfases entre fecha_torneo y match_year "
    "se limitan correctamente al cambio de temporada en diciembre."
)

,valor
observaciones_con_desfase,688
partidos_con_desfase,344
torneos_con_desfase,12
desfase_minimo,1
desfase_maximo,1
mes_minimo,12
mes_maximo,12
dia_minimo,29
dia_maximo,31


Los desfases entre fecha_torneo y match_year se limitan correctamente al cambio de temporada en diciembre.


### 3.5. Validación de las variables fuente de carga

**Qué hago.** Analizo la calidad de `total_juegos` y `total_sets`, que utilizaré posteriormente para medir el volumen competitivo acumulado. También compruebo que cada observación jugador–partido puede contabilizarse como una única participación competitiva.

**Por qué lo hago.** Antes de construir variables históricas necesito asegurarme de que las unidades básicas de carga no contienen valores negativos, valores decimales impropios, ausencias no identificadas o cantidades incompatibles con su interpretación.

**Riesgo metodológico controlado.** Evito propagar errores del partido original a todas las observaciones futuras del jugador. Un valor incorrecto de juegos o sets podría afectar a múltiples variables acumuladas y distorsionar posteriormente el análisis y los modelos.

Para medir la exposición competitiva utilizaré el volumen total del encuentro. Por tanto, cada participación aportará un partido, el total de sets disputados y el total de juegos disputados.

In [11]:
# Compruebo que las variables fuente necesarias están disponibles.
columnas_fuente_carga = [
    "_id_partido",
    "jugador_id",
    "total_juegos",
    "total_sets"
]

columnas_faltantes = [
    columna
    for columna in columnas_fuente_carga
    if columna not in df_check.columns
]

assert not columnas_faltantes, (
    "No encuentro algunas columnas necesarias para construir la carga: "
    f"{columnas_faltantes}"
)


# Convierto las variables de volumen a formato numérico.
# errors='coerce' permite identificar cualquier valor no interpretable.
for columna in ["total_juegos", "total_sets"]:
    df_check[columna] = pd.to_numeric(
        df_check[columna],
        errors="coerce"
    )


# Calculo un resumen inicial de calidad.
resumen_fuentes_carga = pd.DataFrame(
    {
        "tipo_dato": [
            str(df_check["total_juegos"].dtype),
            str(df_check["total_sets"].dtype)
        ],
        "numero_nulos": [
            df_check["total_juegos"].isna().sum(),
            df_check["total_sets"].isna().sum()
        ],
        "porcentaje_nulos": [
            df_check["total_juegos"].isna().mean() * 100,
            df_check["total_sets"].isna().mean() * 100
        ],
        "valor_minimo": [
            df_check["total_juegos"].min(),
            df_check["total_sets"].min()
        ],
        "mediana": [
            df_check["total_juegos"].median(),
            df_check["total_sets"].median()
        ],
        "valor_maximo": [
            df_check["total_juegos"].max(),
            df_check["total_sets"].max()
        ]
    },
    index=["total_juegos", "total_sets"]
)

display(resumen_fuentes_carga)

,tipo_dato,numero_nulos,porcentaje_nulos,valor_minimo,mediana,valor_maximo
total_juegos,int64,0,0.0,0,24.0,183
total_sets,int64,0,0.0,0,2.0,5


In [12]:
import numpy as np


# Compruebo valores negativos.
valores_negativos = pd.Series(
    {
        "total_juegos": (
            df_check["total_juegos"].lt(0).sum()
        ),
        "total_sets": (
            df_check["total_sets"].lt(0).sum()
        )
    },
    name="numero_valores_negativos"
)


# Identifico valores iguales a cero.
# No los considero automáticamente errores porque podrían estar
# relacionados con partidos incompletos o casos especiales.
valores_cero = pd.Series(
    {
        "total_juegos": (
            df_check["total_juegos"].eq(0).sum()
        ),
        "total_sets": (
            df_check["total_sets"].eq(0).sum()
        )
    },
    name="numero_valores_cero"
)


# Compruebo que los valores no ausentes son enteros.
# Utilizo isclose para evitar problemas de representación numérica.
def es_entero_o_nulo(serie, tolerancia=1e-10):
    mascara_no_nula = serie.notna()

    resultado = pd.Series(
        True,
        index=serie.index
    )

    resultado.loc[mascara_no_nula] = np.isclose(
        serie.loc[mascara_no_nula],
        np.round(serie.loc[mascara_no_nula]),
        atol=tolerancia,
        rtol=0
    )

    return resultado


juegos_enteros = es_entero_o_nulo(
    df_check["total_juegos"]
)

sets_enteros = es_entero_o_nulo(
    df_check["total_sets"]
)


resumen_validez_numerica = pd.DataFrame(
    {
        "valores_negativos": valores_negativos,
        "valores_cero": valores_cero,
        "valores_no_enteros": [
            (~juegos_enteros).sum(),
            (~sets_enteros).sum()
        ]
    },
    index=["total_juegos", "total_sets"]
)

display(resumen_validez_numerica)


# Los valores negativos y no enteros sí serían incompatibles
# con la interpretación de las variables.
assert valores_negativos.sum() == 0, (
    "He encontrado valores negativos en las variables de carga."
)

assert juegos_enteros.all(), (
    "He encontrado valores no enteros en total_juegos."
)

assert sets_enteros.all(), (
    "He encontrado valores no enteros en total_sets."
)

,valores_negativos,valores_cero,valores_no_enteros
total_juegos,0,2,0
total_sets,0,514,0


In [13]:
# Selecciono los registros en los que total_juegos o total_sets
# están ausentes o son iguales a cero.
mascara_fuentes_problematicas = (
    df_check["total_juegos"].isna()
    | df_check["total_sets"].isna()
    | df_check["total_juegos"].eq(0)
    | df_check["total_sets"].eq(0)
)


# Incluyo jugador_id porque lo utilizo posteriormente
# para ordenar las dos perspectivas del partido.
columnas_inspeccion_fuentes = [
    "_id_partido",
    "fecha_torneo",
    "match_year",
    "identificador_torneo",
    "nombre_torneo",
    "nivel_torneo",
    "ronda",
    "jugador_id",
    "jugador_nombre",
    "rival_id",
    "rival_nombre",
    "marcador_partido",
    "juegos_ganados",
    "juegos_perdidos",
    "total_juegos",
    "sets_ganados",
    "sets_perdidos",
    "total_sets",
    "porcentaje_juegos_ganados"
]


filas_fuentes_problematicas = (
    df_check.loc[
        mascara_fuentes_problematicas,
        columnas_inspeccion_fuentes
    ]
    .sort_values(
        [
            "fecha_torneo",
            "_id_partido",
            "jugador_id"
        ]
    )
)


print(
    "Número de observaciones con juegos o sets "
    "ausentes o iguales a cero:"
)

print(len(filas_fuentes_problematicas))


print(
    "Número de partidos afectados:"
)

print(
    filas_fuentes_problematicas[
        "_id_partido"
    ].nunique()
)


if not filas_fuentes_problematicas.empty:
    display(filas_fuentes_problematicas)

Número de observaciones con juegos o sets ausentes o iguales a cero:
514
Número de partidos afectados:
257


,_id_partido,fecha_torneo,match_year,identificador_torneo,nombre_torneo,nivel_torneo,ronda,jugador_id,jugador_nombre,rival_id,rival_nombre,marcador_partido,juegos_ganados,juegos_perdidos,total_juegos,sets_ganados,sets_perdidos,total_sets,porcentaje_juegos_ganados
683,2008-D008__5,2008-02-08,2008,2008-D008,Davis Cup G1 R1: SUI vs POL,Davis_Cup,RR,104219,Stephane Bohli,105101,Blazej Koniusz,2-1 RET,2,1,3,0,0,0,0.666667
682,2008-D008__5,2008-02-08,2008,2008-D008,Davis Cup G1 R1: SUI vs POL,Davis_Cup,RR,105101,Blazej Koniusz,104219,Stephane Bohli,2-1 RET,1,2,3,0,0,0,0.333333
1149,2008-506__14,2008-02-18,2008,2008-506,Buenos Aires,ATP,R32,103373,Sergio Roitman,104371,Ivo Minar,4-0 RET,4,0,4,0,0,0,1.000000
1148,2008-506__14,2008-02-18,2008,2008-506,Buenos Aires,ATP,R32,104371,Ivo Minar,103373,Sergio Roitman,4-0 RET,0,4,4,0,0,0,0.000000
2522,2008-425__27,2008-04-28,2008,2008-425,Barcelona,ATP,R32,103428,Juan Ignacio Chela,104043,Marc Lopez,5-1 RET,5,1,6,0,0,0,0.833333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92789,2024-M-DC-2024-FLS-2-M-B-AUS-CZE-01__2,2024-09-12,2024,2024-M-DC-2024-FLS-2-M-B-AUS-CZE-01,Davis Cup Finals RR: AUS vs CZE,Davis_Cup,RR,207830,Tomas Machac,200615,Alexei Popyrin,1-0 RET,0,1,1,0,0,0,0.000000
93277,2024-5014__286,2024-10-02,2024,2024-5014,Shanghai Masters,Masters_1000,R128,106415,Yoshihito Nishioka,212044,Yi Zhou,2-4 RET,4,2,6,0,0,0,0.666667
93276,2024-5014__286,2024-10-02,2024,2024-5014,Shanghai Masters,Masters_1000,R128,212044,Yi Zhou,106415,Yoshihito Nishioka,2-4 RET,2,4,6,0,0,0,0.333333
93584,2024-9410__375,2024-10-14,2024,2024-9410,Almaty,ATP,R16,202320,Beibit Zhukayev,207830,Tomas Machac,3-1 RET,3,1,4,0,0,0,0.750000


In [14]:
# Compruebo la coherencia lógica entre ambas medidas.
juegos_positivos_sets_no_positivos = df_check.loc[
    df_check["total_juegos"].gt(0)
    & (
        df_check["total_sets"].isna()
        | df_check["total_sets"].le(0)
    )
].copy()


sets_positivos_juegos_no_positivos = df_check.loc[
    df_check["total_sets"].gt(0)
    & (
        df_check["total_juegos"].isna()
        | df_check["total_juegos"].le(0)
    )
].copy()


resumen_coherencia_juegos_sets = pd.Series(
    {
        "filas_con_juegos_positivos_y_sets_no_positivos": (
            len(juegos_positivos_sets_no_positivos)
        ),
        "filas_con_sets_positivos_y_juegos_no_positivos": (
            len(sets_positivos_juegos_no_positivos)
        )
    },
    name="numero_anomalias"
)

display(resumen_coherencia_juegos_sets.to_frame())


if not juegos_positivos_sets_no_positivos.empty:
    print(
        "He encontrado registros con juegos positivos "
        "pero sin sets positivos:"
    )

    display(
        juegos_positivos_sets_no_positivos[
            [
                "_id_partido",
                "jugador_nombre",
                "rival_nombre",
                "marcador_partido",
                "total_juegos",
                "total_sets"
            ]
        ].head(20)
    )


if not sets_positivos_juegos_no_positivos.empty:
    print(
        "He encontrado registros con sets positivos "
        "pero sin juegos positivos:"
    )

    display(
        sets_positivos_juegos_no_positivos[
            [
                "_id_partido",
                "jugador_nombre",
                "rival_nombre",
                "marcador_partido",
                "total_juegos",
                "total_sets"
            ]
        ].head(20)
    )

,numero_anomalias
filas_con_juegos_positivos_y_sets_no_positivos,512
filas_con_sets_positivos_y_juegos_no_positivos,0


He encontrado registros con juegos positivos pero sin sets positivos:


,_id_partido,jugador_nombre,rival_nombre,marcador_partido,total_juegos,total_sets
682,2008-D008__5,Blazej Koniusz,Stephane Bohli,2-1 RET,3,0
683,2008-D008__5,Stephane Bohli,Blazej Koniusz,2-1 RET,3,0
1148,2008-506__14,Ivo Minar,Sergio Roitman,4-0 RET,4,0
1149,2008-506__14,Sergio Roitman,Ivo Minar,4-0 RET,4,0
2474,2008-425__3,Feliciano Lopez,Jose Acasuso,5-2 RET,7,0
2475,2008-425__3,Jose Acasuso,Feliciano Lopez,5-2 RET,7,0
2478,2008-425__5,Ivo Minar,Marc Lopez,3-1 RET,4,0
2479,2008-425__5,Marc Lopez,Ivo Minar,3-1 RET,4,0
2522,2008-425__27,Juan Ignacio Chela,Marc Lopez,5-1 RET,6,0
2523,2008-425__27,Marc Lopez,Juan Ignacio Chela,5-1 RET,6,0


In [15]:
# Creo una tabla con una única fila por partido.
df_volumen_partido = (
    df_check[
        [
            "_id_partido",
            "fecha_torneo",
            "match_year",
            "total_juegos",
            "total_sets"
        ]
    ]
    .drop_duplicates(subset="_id_partido")
    .copy()
)


# Compruebo que el número de filas coincide con el número
# de partidos previamente validado.
assert len(df_volumen_partido) == df_check["_id_partido"].nunique(), (
    "La tabla de volumen por partido no contiene "
    "el número esperado de encuentros."
)


resumen_partidos_fuente = pd.Series(
    {
        "numero_partidos": len(df_volumen_partido),
        "partidos_con_total_juegos_ausente": (
            df_volumen_partido["total_juegos"].isna().sum()
        ),
        "partidos_con_total_sets_ausente": (
            df_volumen_partido["total_sets"].isna().sum()
        ),
        "partidos_con_total_juegos_cero": (
            df_volumen_partido["total_juegos"].eq(0).sum()
        ),
        "partidos_con_total_sets_cero": (
            df_volumen_partido["total_sets"].eq(0).sum()
        ),
        "partidos_con_juegos_y_sets_validos": (
            (
                df_volumen_partido["total_juegos"].gt(0)
                & df_volumen_partido["total_sets"].gt(0)
            ).sum()
        )
    },
    name="valor"
)

display(resumen_partidos_fuente.to_frame())

,valor
numero_partidos,47016
partidos_con_total_juegos_ausente,0
partidos_con_total_sets_ausente,0
partidos_con_total_juegos_cero,1
partidos_con_total_sets_cero,257
partidos_con_juegos_y_sets_validos,46759


#### Interpretación de los partidos sin sets completados

La revisión identifica partidos con `total_sets = 0`, pero esta situación no implica necesariamente ausencia de actividad competitiva. En la mayoría de estos casos existen juegos efectivamente disputados, aunque el encuentro finalizó antes de completarse ningún set, principalmente por retirada.

Por este motivo, estos partidos se conservan cuando `total_juegos > 0`, ya que representan exposición competitiva observable y pueden contribuir a las variables de carga basadas en juegos y partidos. Únicamente se excluye posteriormente el caso en el que tanto `total_juegos` como `total_sets` son iguales a cero, al no existir volumen competitivo observable.

In [16]:
# Muestro los partidos con más juegos disputados.
partidos_mayor_numero_juegos = (
    df_check[
        [
            "_id_partido",
            "fecha_torneo",
            "nombre_torneo",
            "nivel_torneo",
            "ronda",
            "numero_maximo_sets",
            "jugador_nombre",
            "rival_nombre",
            "marcador_partido",
            "total_juegos",
            "total_sets"
        ]
    ]
    .drop_duplicates(subset="_id_partido")
    .nlargest(15, "total_juegos")
)

display(partidos_mayor_numero_juegos)


# Muestro los partidos con más sets registrados.
partidos_mayor_numero_sets = (
    df_check[
        [
            "_id_partido",
            "fecha_torneo",
            "nombre_torneo",
            "nivel_torneo",
            "ronda",
            "numero_maximo_sets",
            "jugador_nombre",
            "rival_nombre",
            "marcador_partido",
            "total_juegos",
            "total_sets"
        ]
    ]
    .drop_duplicates(subset="_id_partido")
    .nlargest(15, "total_sets")
)

display(partidos_mayor_numero_sets)

,_id_partido,fecha_torneo,nombre_torneo,nivel_torneo,ronda,numero_maximo_sets,jugador_nombre,rival_nombre,marcador_partido,total_juegos,total_sets
15788,2010-540__60,2010-06-21,Wimbledon,Grand_Slam,R128,5,John Isner,Nicolas Mahut,6-4 3-6 6-7(7) 7-6(3) 70-68,183,5
63096,2018-540__224,2018-07-02,Wimbledon,Grand_Slam,SF,5,John Isner,Kevin Anderson,7-6(6) 6-7(5) 6-7(9) 6-4 26-24,99,5
54026,2017-580__152,2017-01-16,Australian Open,Grand_Slam,R128,5,Horacio Zeballos,Ivo Karlovic,6-7(6) 3-6 7-5 6-2 22-20,84,5
11180,2009-D013__1,2009-09-18,Davis Cup WG SF: CRO vs CZE,Davis_Cup,RR,5,Ivo Karlovic,Radek Stepanek,6-7(5) 7-6(5) 7-6(6) 6-7(2) 16-14,82,5
28062,2012-540__107,2012-06-25,Wimbledon,Grand_Slam,R32,5,Marin Cilic,Sam Querrey,7-6(6) 6-4 6-7(2) 6-7(3) 17-15,81,5
51284,2016-540__209,2016-06-27,Wimbledon,Grand_Slam,R32,5,Jo-Wilfried Tsonga,John Isner,6-7(3) 3-6 7-6(5) 6-2 19-17,79,5
43198,2015-D005__4,2015-03-06,Davis Cup WG R1: ARG vs BRA,Davis_Cup,RR,5,Joao Souza,Leonardo Mayer,7-6(4) 7-6(5) 5-7 5-7 15-13,78,5
1960,2008-D036__1,2008-04-11,Davis Cup G1 R2: SVK vs GEO,Davis_Cup,RR,5,Irakli Labadze,Lukas Lacko,4-6 7-6(6) 3-6 6-3 19-17,77,5
9818,2009-540__127,2009-06-22,Wimbledon,Grand_Slam,F,5,Andy Roddick,Roger Federer,5-7 7-6(6) 7-6(5) 3-6 16-14,77,5
27484,2012-520__84,2012-05-27,Roland Garros,Grand_Slam,R64,5,John Isner,Paul Henri Mathieu,6-7(2) 6-4 6-4 3-6 18-16,76,5


,_id_partido,fecha_torneo,nombre_torneo,nivel_torneo,ronda,numero_maximo_sets,jugador_nombre,rival_nombre,marcador_partido,total_juegos,total_sets
310,2008-580__3,2008-01-14,Australian Open,Grand_Slam,R128,5,Janko Tipsarevic,Joseph Sirianni,7-5 6-2 6-7(6) 0-6 6-0,45,5
324,2008-580__10,2008-01-14,Australian Open,Grand_Slam,R128,5,Fabio Fognini,Michael Russell,6-1 4-6 6-2 3-6 6-3,43,5
340,2008-580__18,2008-01-14,Australian Open,Grand_Slam,R128,5,Rajeev Ram,Simone Bolelli,7-6(5) 3-6 6-4 2-6 6-3,49,5
344,2008-580__20,2008-01-14,Australian Open,Grand_Slam,R128,5,Dmitry Tursunov,Xavier Malisse,6-7(1) 5-7 6-2 6-1 6-3,49,5
362,2008-580__29,2008-01-14,Australian Open,Grand_Slam,R128,5,Radek Stepanek,Vincent Spadea,2-6 2-6 7-5 6-2 6-3,45,5
380,2008-580__38,2008-01-14,Australian Open,Grand_Slam,R128,5,Alejandro Falla,Kevin Anderson,5-7 7-5 6-7(8) 6-2 7-5,57,5
382,2008-580__39,2008-01-14,Australian Open,Grand_Slam,R128,5,Juan Pablo Brzezicki,Sam Warburg,2-6 6-2 6-3 2-6 6-4,43,5
396,2008-580__46,2008-01-14,Australian Open,Grand_Slam,R128,5,Marc Gicquel,Yen Hsun Lu,6-3 4-6 6-7(6) 6-4 6-2,50,5
398,2008-580__47,2008-01-14,Australian Open,Grand_Slam,R128,5,Jose Acasuso,Nicolas Mahut,7-6(2) 5-7 6-2 3-6 7-5,54,5
410,2008-580__53,2008-01-14,Australian Open,Grand_Slam,R128,5,Frank Dancevic,Jarkko Nieminen,6-3 6-1 5-7 2-6 6-1,43,5


### 3.6. Validación de la duración de los partidos

**Qué hago.** Analizo la calidad de `duracion_partido_minutos` mediante su cobertura global, su evolución por temporada y nivel de torneo, la presencia de valores imposibles y su coherencia con los juegos y sets disputados.

**Por qué lo hago.** La duración podría representar una dimensión adicional del volumen competitivo que no queda completamente recogida por el número de partidos, sets o juegos. Sin embargo, solo resultaría adecuada si su cobertura fuese suficientemente amplia y homogénea.

**Separación temporal.** Como esta auditoría contribuye a decidir si la duración debe utilizarse posteriormente para construir variables de carga, se excluye la temporada 2024 reservada para la evaluación temporal externa. Se mantienen los años buffer porque forman parte del historial operativo necesario para reconstruir la actividad previa de las temporadas de estudio.

La duración no se utiliza para predecir el partido en el que se genera; únicamente podría incorporarse como información histórica procedente de participaciones anteriores.

In [17]:
# Compruebo que la variable de duración está disponible.
assert "duracion_partido_minutos" in df_check.columns, (
    "No encuentro la columna duracion_partido_minutos."
)


# La auditoría que sustenta la decisión sobre esta variable
# excluye 2024, reservado para la evaluación temporal externa.
columnas_duracion = [
    "_id_partido",
    "fecha_torneo",
    "match_year",
    "identificador_torneo",
    "nombre_torneo",
    "nivel_torneo",
    "ronda",
    "numero_maximo_sets",
    "marcador_partido",
    "total_juegos",
    "total_sets",
    "duracion_partido_minutos"
]

df_duracion_partido = (
    df_check.loc[
        df_check["es_validacion_temporal"].eq(0),
        columnas_duracion
    ]
    .drop_duplicates(subset="_id_partido")
    .copy()
)


# Convierto la duración a formato numérico.
# Los valores no interpretables se transforman en NaN
# para poder identificarlos durante la auditoría.
df_duracion_partido["duracion_partido_minutos"] = pd.to_numeric(
    df_duracion_partido["duracion_partido_minutos"],
    errors="coerce"
)


assert 2024 not in df_duracion_partido["match_year"].unique(), (
    "La evaluación temporal externa de 2024 no debe intervenir "
    "en la auditoría utilizada para decidir sobre la duración."
)

numero_partidos_esperado = (
    df_check.loc[
        df_check["es_validacion_temporal"].eq(0),
        "_id_partido"
    ]
    .nunique()
)

assert len(df_duracion_partido) == numero_partidos_esperado, (
    "La tabla de duración no contiene el número esperado de partidos."
)


# Calculo el resumen general de cobertura.
numero_partidos = len(df_duracion_partido)

numero_duraciones_nulas = (
    df_duracion_partido["duracion_partido_minutos"]
    .isna()
    .sum()
)

numero_duraciones_disponibles = (
    df_duracion_partido["duracion_partido_minutos"]
    .notna()
    .sum()
)

resumen_duracion = pd.Series(
    {
        "numero_partidos": numero_partidos,
        "partidos_con_duracion_disponible": (
            numero_duraciones_disponibles
        ),
        "partidos_con_duracion_ausente": (
            numero_duraciones_nulas
        ),
        "porcentaje_cobertura": (
            numero_duraciones_disponibles
            / numero_partidos
            * 100
        ),
        "porcentaje_ausencia": (
            numero_duraciones_nulas
            / numero_partidos
            * 100
        ),
        "duracion_minima_disponible": (
            df_duracion_partido[
                "duracion_partido_minutos"
            ].min()
        ),
        "duracion_mediana": (
            df_duracion_partido[
                "duracion_partido_minutos"
            ].median()
        ),
        "duracion_maxima": (
            df_duracion_partido[
                "duracion_partido_minutos"
            ].max()
        )
    },
    name="valor"
)

display(resumen_duracion.to_frame())

,valor
numero_partidos,43961.000000
partidos_con_duracion_disponible,39376.000000
partidos_con_duracion_ausente,4585.000000
porcentaje_cobertura,89.570301
porcentaje_ausencia,10.429699
duracion_minima_disponible,4.000000
duracion_mediana,102.000000
duracion_maxima,1146.000000


In [18]:
# Identifico duraciones negativas.
duraciones_negativas = df_duracion_partido.loc[
    df_duracion_partido[
        "duracion_partido_minutos"
    ].lt(0)
].copy()


# Identifico duraciones iguales a cero.
duraciones_cero = df_duracion_partido.loc[
    df_duracion_partido[
        "duracion_partido_minutos"
    ].eq(0)
].copy()


# Identifico duraciones no enteras.
# No las considero automáticamente erróneas,
# pero quiero conocer su frecuencia.
mascara_duracion_disponible = (
    df_duracion_partido[
        "duracion_partido_minutos"
    ].notna()
)

duraciones_no_enteras = df_duracion_partido.loc[
    mascara_duracion_disponible
    & (
        df_duracion_partido[
            "duracion_partido_minutos"
        ]
        % 1
    ).ne(0)
].copy()


resumen_validez_duracion = pd.Series(
    {
        "partidos_con_duracion_negativa": (
            len(duraciones_negativas)
        ),
        "partidos_con_duracion_cero": (
            len(duraciones_cero)
        ),
        "partidos_con_duracion_no_entera": (
            len(duraciones_no_enteras)
        )
    },
    name="numero_partidos"
)

display(resumen_validez_duracion.to_frame())


# Una duración negativa sería incompatible
# con la interpretación de la variable.
assert duraciones_negativas.empty, (
    "He encontrado partidos con duración negativa."
)

,numero_partidos
partidos_con_duracion_negativa,0
partidos_con_duracion_cero,0
partidos_con_duracion_no_entera,0


In [19]:
if not duraciones_cero.empty:
    print(
        "He encontrado partidos con duración igual a cero:"
    )

    display(
        duraciones_cero[
            [
                "_id_partido",
                "fecha_torneo",
                "match_year",
                "nombre_torneo",
                "nivel_torneo",
                "ronda",
                "marcador_partido",
                "total_juegos",
                "total_sets",
                "duracion_partido_minutos"
            ]
        ]
        .sort_values(
            ["fecha_torneo", "_id_partido"]
        )
    )
else:
    print(
        "No he encontrado partidos con duración igual a cero."
    )

No he encontrado partidos con duración igual a cero.


In [20]:
duracion_positiva_juegos_cero = df_duracion_partido.loc[
    df_duracion_partido[
        "duracion_partido_minutos"
    ].gt(0)
    & df_duracion_partido["total_juegos"].eq(0)
].copy()


juegos_positivos_duracion_cero = df_duracion_partido.loc[
    df_duracion_partido["total_juegos"].gt(0)
    & df_duracion_partido[
        "duracion_partido_minutos"
    ].eq(0)
].copy()


duracion_positiva_sets_cero = df_duracion_partido.loc[
    df_duracion_partido[
        "duracion_partido_minutos"
    ].gt(0)
    & df_duracion_partido["total_sets"].eq(0)
].copy()


sets_positivos_duracion_cero = df_duracion_partido.loc[
    df_duracion_partido["total_sets"].gt(0)
    & df_duracion_partido[
        "duracion_partido_minutos"
    ].eq(0)
].copy()


resumen_coherencia_duracion = pd.Series(
    {
        "duracion_positiva_con_cero_juegos": (
            len(duracion_positiva_juegos_cero)
        ),
        "juegos_positivos_con_duracion_cero": (
            len(juegos_positivos_duracion_cero)
        ),
        "duracion_positiva_con_cero_sets": (
            len(duracion_positiva_sets_cero)
        ),
        "sets_positivos_con_duracion_cero": (
            len(sets_positivos_duracion_cero)
        )
    },
    name="numero_partidos"
)

display(resumen_coherencia_duracion.to_frame())

,numero_partidos
duracion_positiva_con_cero_juegos,0
juegos_positivos_con_duracion_cero,0
duracion_positiva_con_cero_sets,224
sets_positivos_con_duracion_cero,0


In [21]:
cobertura_duracion_temporada = (
    df_duracion_partido
    .groupby("match_year")
    .agg(
        numero_partidos=(
            "_id_partido",
            "size"
        ),
        duraciones_disponibles=(
            "duracion_partido_minutos",
            "count"
        ),
        duracion_mediana=(
            "duracion_partido_minutos",
            "median"
        ),
        duracion_media=(
            "duracion_partido_minutos",
            "mean"
        ),
        duracion_maxima=(
            "duracion_partido_minutos",
            "max"
        )
    )
)


cobertura_duracion_temporada[
    "duraciones_ausentes"
] = (
    cobertura_duracion_temporada[
        "numero_partidos"
    ]
    - cobertura_duracion_temporada[
        "duraciones_disponibles"
    ]
)


cobertura_duracion_temporada[
    "porcentaje_cobertura"
] = (
    cobertura_duracion_temporada[
        "duraciones_disponibles"
    ]
    / cobertura_duracion_temporada[
        "numero_partidos"
    ]
    * 100
)


cobertura_duracion_temporada = (
    cobertura_duracion_temporada[
        [
            "numero_partidos",
            "duraciones_disponibles",
            "duraciones_ausentes",
            "porcentaje_cobertura",
            "duracion_mediana",
            "duracion_media",
            "duracion_maxima"
        ]
    ]
)


display(
    cobertura_duracion_temporada.round(2)
)

,numero_partidos,duraciones_disponibles,duraciones_ausentes,porcentaje_cobertura,duracion_mediana,duracion_media,duracion_maxima
match_year,,,,,,,
2008,3102,2765,337,89.14,97.0,105.39,312.0
2009,3069,2727,342,88.86,101.0,107.54,310.0
2010,3013,2686,327,89.15,100.0,107.03,665.0
2011,3000,2687,313,89.57,100.0,108.32,288.0
2012,2990,2680,310,89.63,101.0,110.29,353.0
2013,2923,2613,310,89.39,96.0,103.84,302.0
2014,2874,2574,300,89.56,98.0,104.38,272.0
2015,2933,1362,1571,46.44,97.5,105.13,252.0
2016,2920,2903,17,99.42,101.0,109.34,1146.0


In [22]:
cobertura_duracion_nivel = (
    df_duracion_partido
    .groupby(
        "nivel_torneo",
        dropna=False
    )
    .agg(
        numero_partidos=(
            "_id_partido",
            "size"
        ),
        duraciones_disponibles=(
            "duracion_partido_minutos",
            "count"
        ),
        duracion_mediana=(
            "duracion_partido_minutos",
            "median"
        ),
        duracion_media=(
            "duracion_partido_minutos",
            "mean"
        ),
        duracion_maxima=(
            "duracion_partido_minutos",
            "max"
        )
    )
)


cobertura_duracion_nivel[
    "duraciones_ausentes"
] = (
    cobertura_duracion_nivel[
        "numero_partidos"
    ]
    - cobertura_duracion_nivel[
        "duraciones_disponibles"
    ]
)


cobertura_duracion_nivel[
    "porcentaje_cobertura"
] = (
    cobertura_duracion_nivel[
        "duraciones_disponibles"
    ]
    / cobertura_duracion_nivel[
        "numero_partidos"
    ]
    * 100
)


cobertura_duracion_nivel = (
    cobertura_duracion_nivel
    .sort_values(
        "porcentaje_cobertura",
        ascending=True
    )
)


display(
    cobertura_duracion_nivel.round(2)
)

,numero_partidos,duraciones_disponibles,duracion_mediana,duracion_media,duracion_maxima,duraciones_ausentes,porcentaje_cobertura
nivel_torneo,,,,,,,
Davis_Cup,3982,974,110.0,119.78,356.0,3008,24.46
ATP_Finals,269,245,98.0,103.80,218.0,24,91.08
Grand_Slam,7592,7220,144.0,151.51,665.0,372,95.10
ATP,23679,22739,93.0,98.79,1146.0,940,96.03
Masters_1000,8439,8198,96.0,101.84,244.0,241,97.14


In [23]:
percentiles_duracion = (
    df_duracion_partido[
        "duracion_partido_minutos"
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

display(
    percentiles_duracion.to_frame(
        name="duracion_partido_minutos"
    )
)

,duracion_partido_minutos
count,39376.000000
mean,109.640339
std,42.138758
min,4.000000
1%,41.000000
5%,58.000000
25%,79.000000
50%,102.000000
75%,133.000000
90%,163.000000


In [24]:
partidos_mayor_duracion = (
    df_duracion_partido
    .nlargest(
        20,
        "duracion_partido_minutos"
    )
    [
        [
            "_id_partido",
            "fecha_torneo",
            "nombre_torneo",
            "nivel_torneo",
            "ronda",
            "numero_maximo_sets",
            "marcador_partido",
            "total_juegos",
            "total_sets",
            "duracion_partido_minutos"
        ]
    ]
)

display(partidos_mayor_duracion)

,_id_partido,fecha_torneo,nombre_torneo,nivel_torneo,ronda,numero_maximo_sets,marcador_partido,total_juegos,total_sets,duracion_partido_minutos
48072,2016-M001__294,2016-01-11,Sydney,ATP,QF,3,7-6(5) 6-3,22,2,1146.0
56030,2017-0308__297,2017-05-01,Munich,ATP,QF,3,6-4 3-6 6-2,27,3,987.0
15788,2010-540__60,2010-06-21,Wimbledon,Grand_Slam,R128,5,6-4 3-6 6-7(7) 7-6(3) 70-68,183,5,665.0
63096,2018-540__224,2018-07-02,Wimbledon,Grand_Slam,SF,5,7-6(6) 6-7(5) 6-7(9) 6-4 26-24,99,5,396.0
61274,2018-M-DC-2018-G1-AO-M-PAK-UZB-01__1,2018-04-06,Davis Cup G1 R2: PAK vs UZB,Davis_Cup,RR,3,7-6(17) 4-1 RET,18,1,356.0
24896,2012-580__127,2012-01-16,Australian Open,Grand_Slam,F,5,5-7 6-4 6-2 6-7(5) 7-5,55,5,353.0
82490,2023-580__192,2023-01-16,Australian Open,Grand_Slam,R64,5,4-6 6-7(4) 7-6(5) 6-3 7-5,57,5,345.0
49098,2016-M-DC-2016-G2-AM-M-PAR-VEN-01__4,2016-03-04,Davis Cup G2 R1: PAR vs VEN,Davis_Cup,RR,3,6-1 6-0,13,2,344.0
27484,2012-520__84,2012-05-27,Roland Garros,Grand_Slam,R64,5,6-7(2) 6-4 6-4 3-6 18-16,76,5,341.0
28062,2012-540__107,2012-06-25,Wimbledon,Grand_Slam,R32,5,7-6(6) 6-4 6-7(2) 6-7(3) 17-15,81,5,331.0


In [25]:
mascara_ratio_calculable = (
    df_duracion_partido[
        "duracion_partido_minutos"
    ].gt(0)
    & df_duracion_partido[
        "total_juegos"
    ].gt(0)
)


df_duracion_partido[
    "minutos_por_juego_auditoria"
] = np.nan


df_duracion_partido.loc[
    mascara_ratio_calculable,
    "minutos_por_juego_auditoria"
] = (
    df_duracion_partido.loc[
        mascara_ratio_calculable,
        "duracion_partido_minutos"
    ]
    / df_duracion_partido.loc[
        mascara_ratio_calculable,
        "total_juegos"
    ]
)


display(
    df_duracion_partido[
        "minutos_por_juego_auditoria"
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.50,
            0.95,
            0.99
        ]
    )
    .to_frame()
)

,minutos_por_juego_auditoria
count,39376.000000
mean,4.310536
std,0.711775
min,0.781818
1%,3.058824
5%,3.375000
50%,4.260870
95%,5.387097
99%,6.000000
max,52.090909


#### Conclusión de la auditoría de la duración

La revisión de `duracion_partido_minutos`, realizada sin utilizar la temporada 2024 reservada para la evaluación temporal externa, muestra que la variable no presenta una cobertura completamente homogénea a lo largo del periodo operativo y que existen algunos valores extremos que requerirían una depuración específica antes de utilizarla como medida cuantitativa de carga.

Por este motivo, la duración se mantiene únicamente como variable auditada y descriptiva, pero no se utiliza posteriormente para construir las variables de carga competitiva acumulada. Estas se basan en medidas de volumen observables y más homogéneas, principalmente juegos, sets, partidos y torneos disputados.

### Tratamiento de partidos sin volumen competitivo observable

Se identifican los partidos con `total_juegos = 0`, ya que en estos casos no existe volumen competitivo medible en términos de juegos disputados. Estos registros no aportan información útil para calcular el porcentaje de juegos ganados ni para construir variables de carga competitiva acumulada.

La exclusión se aplica al nivel de partido completo, eliminando conjuntamente las dos observaciones recíprocas jugador–partido. Esta decisión no implica eliminar de forma general los partidos con retirada (`RET`), ya que aquellos con juegos disputados sí representan exposición competitiva observable y se conservan en el análisis.

In [26]:
# Creo una copia de trabajo sin modificar el dataset original.
df_work = df_master.copy(deep=True)


# Construyo el identificador estable de partido.
df_work["_id_partido"] = (
    df_work["identificador_torneo"]
    .astype("string")
    .str.strip()
    + "__"
    + pd.to_numeric(
        df_work["numero_partido"],
        errors="raise"
    )
    .astype("Int64")
    .astype("string")
)

In [27]:

# Identifico los partidos sin volumen competitivo observable.
ids_partidos_volumen_nulo = (
    df_work.loc[
        df_work["total_juegos"].eq(0),
        "_id_partido"
    ]
    .unique()
)

print("Partidos con volumen nulo:", ids_partidos_volumen_nulo)
print("Número de partidos afectados:", len(ids_partidos_volumen_nulo))


# Verifico que existe al menos un partido sin volumen competitivo observable.
assert len(ids_partidos_volumen_nulo) >= 1, (
    "El número de partidos con total_juegos igual a cero "
    "no coincide con el único caso auditado."
)


# Extraigo las dos observaciones correspondientes.
filas_volumen_nulo = df_work.loc[
    df_work["_id_partido"].isin(ids_partidos_volumen_nulo)
].copy()


# Compruebo que el partido contiene exactamente
# sus dos perspectivas jugador-partido.
assert len(filas_volumen_nulo) == 2, (
    "El partido con volumen nulo no contiene exactamente "
    "sus dos perspectivas."
)


# Compruebo que realmente no existe volumen competitivo
# ni en juegos ni en sets.
assert filas_volumen_nulo["total_juegos"].eq(0).all(), (
    "El partido identificado no presenta total_juegos = 0 "
    "en ambas perspectivas."
)

assert filas_volumen_nulo["total_sets"].eq(0).all(), (
    "El partido identificado no presenta total_sets = 0 "
    "en ambas perspectivas."
)


# Al no existir juegos disputados, el porcentaje de juegos
# ganados no puede calcularse.
assert filas_volumen_nulo[
    "porcentaje_juegos_ganados"
].isna().all(), (
    "El objetivo debería ser ausente en el partido "
    "sin volumen competitivo observable."
)

print(
    "El único partido sin volumen competitivo observable "
    "ha sido identificado correctamente."
)

Partidos con volumen nulo: <StringArray>
['2013-338__14']
Length: 1, dtype: string
Número de partidos afectados: 1
El único partido sin volumen competitivo observable ha sido identificado correctamente.


In [28]:
# Elimino las dos perspectivas del partido con volumen nulo.
df_work = (
    df_work.loc[
        ~df_work["_id_partido"].isin(
            ids_partidos_volumen_nulo
        )
    ]
    .copy()
    .reset_index(drop=True)
)

In [29]:
# Compruebo las nuevas dimensiones de forma dinámica.

filas_esperadas = (
    len(df_master)
    - 2 * len(ids_partidos_volumen_nulo)
)

partidos_esperados = (
    df_master[["identificador_torneo", "numero_partido"]]
    .drop_duplicates()
    .shape[0]
    - len(ids_partidos_volumen_nulo)
)

assert df_work.shape[0] == filas_esperadas, (
    "El número de filas después de la exclusión no es el esperado."
)

assert df_work["_id_partido"].nunique() == partidos_esperados, (
    "El número de partidos después de la exclusión no es el esperado."
)

assert not df_work["total_juegos"].eq(0).any(), (
    "Todavía existen observaciones con total_juegos igual a cero."
)

assert (
    df_work.groupby("_id_partido")
    .size()
    .eq(2)
    .all()
), (
    "Algún partido no conserva exactamente dos perspectivas."
)

assert (
    df_work["es_buffer_carga"]
    + df_work["es_desarrollo_modelo"]
    + df_work["es_validacion_temporal"]
).eq(1).all()

assert (
    df_work["es_desarrollo_modelo"]
    + df_work["es_validacion_temporal"]
).eq(df_work["es_muestra_estudio"]).all()

print(
    "He eliminado correctamente las dos perspectivas de los partidos "
    "sin volumen competitivo observable."
)

print("Dimensiones de df_work:", df_work.shape)
print("Número de partidos:", df_work["_id_partido"].nunique())

He eliminado correctamente las dos perspectivas de los partidos sin volumen competitivo observable.
Dimensiones de df_work: (94030, 79)
Número de partidos: 47015


### Exportación del dataset depurado

Después de completar las comprobaciones técnicas, guardo una nueva versión del conjunto de datos que excluye las dos observaciones correspondientes al único partido con volumen competitivo nulo.

No sobrescribo el archivo original, con el objetivo de conservar la trazabilidad del procesamiento. La nueva versión incluye `_id_partido`, que utilizaré como identificador estable para vincular las dos perspectivas de cada encuentro y desarrollar las fases posteriores.

El archivo exportado constituye el punto de partida del notebook principal de construcción de las variables de carga competitiva acumulada.

In [30]:
from pathlib import Path

RUTA_DATOS_LIMPIOS = Path("df_jugador_limpio.csv")

assert not df_work["total_juegos"].eq(0).any(), (
    "Todavía existe alguna observación con total_juegos igual a cero."
)

assert (
    df_work.groupby("_id_partido")
    .size()
    .eq(2)
    .all()
), (
    "Algún partido no contiene exactamente dos observaciones."
)

assert (
    df_work["es_buffer_carga"]
    + df_work["es_desarrollo_modelo"]
    + df_work["es_validacion_temporal"]
).eq(1).all()

assert (
    df_work["es_desarrollo_modelo"]
    + df_work["es_validacion_temporal"]
).eq(df_work["es_muestra_estudio"]).all()

assert 2020 not in df_work["match_year"].unique()

assert df_work.loc[
    df_work["match_year"].isin([2008, 2021]),
    "es_buffer_carga"
].eq(1).all()

assert df_work.loc[
    df_work["match_year"].eq(2024),
    "es_validacion_temporal"
].eq(1).all()

df_work.to_csv(
    RUTA_DATOS_LIMPIOS,
    index=False,
    encoding="utf-8-sig"
)

print("Dataset limpio guardado correctamente.")
print("Ruta:", RUTA_DATOS_LIMPIOS.resolve())
print("Dimensiones:", df_work.shape)
print("Partidos:", df_work["_id_partido"].nunique())

Dataset limpio guardado correctamente.
Ruta: C:\Users\Usuario\Desktop\TFG\Datos\tennis_atp-master\interim\df_jugador_limpio.csv
Dimensiones: (94030, 79)
Partidos: 47015


## Conclusión

Las comprobaciones realizadas confirman la coherencia estructural, recíproca y temporal del dataset jugador–partido utilizado en las etapas posteriores del proyecto.

Se verificó que cada encuentro mantiene sus dos perspectivas correctamente vinculadas, que los juegos y sets son coherentes entre jugador y rival, que `porcentaje_juegos_ganados` reproduce la definición establecida y que la separación entre temporadas de carga, desarrollo y evaluación temporal se conserva correctamente.

La auditoría permitió además identificar y excluir el único encuentro sin volumen competitivo observable, manteniendo los partidos incompletos en los que sí existen juegos disputados.

El dataset resultante, `df_jugador_limpio.csv`, contiene la estructura depurada que se utilizará a continuación para construir las variables de carga competitiva acumulada.